# NB02 — Reading Methods Sections Critically

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours  

---

## Motivation

A methods section answers one question: could someone else reproduce this? If the answer is no, the paper's claims are not fully verifiable, regardless of how impressive the results look. The gap between what a methods section says and what would actually be needed to reproduce the work is one of the defining problems in computational science today.

Learning to read methods critically is a Track A skill (spotting weak methods in papers you cite prevents embarrassing citations), a Track B skill (PhD supervisors evaluate whether you can distinguish rigorous from sloppy work), and a scientific integrity skill.

## 1. What a Methods Section Must Contain

A fully reproducible computational biology methods section must specify:

| Category | Must include | Often missing |
|----------|-------------|---------------|
| **Software** | Name, version number, URL or DOI | Version number ("we used STAR" without version) |
| **Parameters** | All non-default parameters used | "default parameters" with no defaults stated |
| **Data** | Accession numbers (GEO, SRA, ENA, UniProt) | Data described vaguely without accession |
| **Randomness** | Random seed for any stochastic step | Seed omitted (results not reproducible) |
| **Hardware** | CPU/GPU specs for runtime claims | Hardware omitted when runtime is reported |
| **Statistics** | Test used, correction method, effect size | p-values without test name or correction |
| **Code** | Repository URL, commit hash or release | "code available upon request" |
| **Environment** | OS, language version, dependency list | Language version omitted |

## 2. The Replication Gap

The **replication gap** is the distance between what the methods section states and what an independent researcher would actually need to reproduce the work. It has three forms:

1. **Specification gap:** A required parameter or software version is not mentioned.
2. **Accessibility gap:** The requirement is mentioned but the resource is not accessible (e.g., "data available on request" for a corresponding author who has since moved institutions).
3. **Comprehension gap:** The requirement is stated but ambiguously ("we performed standard preprocessing" — what is standard?).

## 3. Red Flags

These phrases in a methods section deserve immediate `?` marks:

- `"standard parameters"` — whose standard? Version-specific defaults change.
- `"publicly available code"` — without a URL or commit hash, this is unfindable.
- `"similar to prior work"` — without a citation, this is circular.
- `"data available upon reasonable request"` — not publicly available; access depends on the author responding.
- `"we selected the best-performing model"` — best on what data? How was selection done? Potential cherry-picking.
- `"after quality filtering"` — what thresholds? Applied to what metric?
- `"convergence was assessed visually"` — subjective criterion with no reproducible threshold.

## 4. Python Implementation: MethodsChecker

The `MethodsChecker` takes a methods section text and flags reproducibility issues.

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Dict


@dataclass
class MethodsFlag:
    category: str
    severity: str  # 'critical', 'major', 'minor'
    pattern_matched: str
    message: str


class MethodsChecker:
    """Scan a methods section text for reproducibility red flags."""

    RED_FLAG_PATTERNS = [
        # (regex, category, severity, message)
        (r'standard parameters?', 'Parameters', 'major',
         'Vague: "standard parameters" — specify all non-default values explicitly'),
        (r'default (settings|parameters|values)', 'Parameters', 'major',
         'Vague: what are the defaults? Software version determines defaults'),
        (r'publicly available code', 'Code', 'critical',
         'Missing URL or commit hash — "publicly available" is not findable without a link'),
        (r'code (is |are |was |were )?available (on|upon) (request|reasonable request)', 'Code', 'critical',
         'Code is not public — not reproducible without author response'),
        (r'similar to (prior|previous|our previous) work', 'Methods', 'major',
         'Vague reference — must cite specific paper and state what is similar'),
        (r'after (quality )?filtering', 'Data', 'major',
         'Filtering criteria not stated — what thresholds? What metric?'),
        (r'convergence was (assessed|checked|verified) visually', 'Statistics', 'major',
         'Subjective criterion — state a reproducible convergence threshold'),
        (r'best[- ]performing model', 'Statistics', 'major',
         'Model selection criterion not stated — best on what data, using what metric?'),
        (r'data (are |is )?available (on|upon) (request|reasonable request)', 'Data', 'critical',
         'Data not publicly deposited — should have accession number (GEO/SRA/ENA)'),
    ]

    REQUIRED_ELEMENTS = {
        'software_version': (
            r'v\d+\.\d+|version \d+\.\d+|\d+\.\d+\.\d+',
            'Software version number (e.g., v2.1.3)'
        ),
        'random_seed': (
            r'random[_ ]?seed|set\.seed|numpy\.random\.seed|torch\.manual_seed|seed\s*=\s*\d+',
            'Random seed (for any stochastic step)'
        ),
        'accession_number': (
            r'GSE\d+|SRR\d+|SRX\d+|PRJNA\d+|EGAS\d+|E-MTAB-\d+|PXD\d+',
            'Data accession number (GEO/SRA/ENA/PRIDE)'
        ),
        'statistical_test': (
            r't-test|wilcoxon|mann[- ]whitney|anova|chi[- ]?square|fisher|kruskal|spearman|pearson',
            'Name of statistical test used'
        ),
        'code_repository': (
            r'github\.com|gitlab\.com|bitbucket\.org|zenodo\.org|doi\.org',
            'Code repository URL'
        ),
    }

    def check(self, text: str) -> Dict:
        text_lower = text.lower()
        flags = []

        # Check red flag patterns
        for pattern, category, severity, message in self.RED_FLAG_PATTERNS:
            if re.search(pattern, text_lower):
                match = re.search(pattern, text_lower).group()
                flags.append(MethodsFlag(category, severity, match, message))

        # Check for required elements
        missing = []
        present = []
        for element, (pattern, description) in self.REQUIRED_ELEMENTS.items():
            if re.search(pattern, text_lower):
                present.append(element)
            else:
                missing.append((element, description))

        # Compute completeness score (0-100)
        completeness = int(100 * len(present) / len(self.REQUIRED_ELEMENTS))

        return {
            'flags': flags,
            'present_elements': present,
            'missing_elements': missing,
            'completeness_score': completeness,
            'critical_flags': sum(1 for f in flags if f.severity == 'critical'),
            'major_flags': sum(1 for f in flags if f.severity == 'major'),
        }


# --- Three synthetic methods sections of increasing quality ---

methods_weak = """
We performed RNA-seq analysis using standard parameters. After quality filtering,
reads were aligned to the reference genome. Differentially expressed genes were
identified using a similar approach to prior work. Code is available upon reasonable
request from the corresponding author.
"""

methods_medium = """
We performed RNA-seq analysis using STAR v2.7.10a with default settings for alignment.
Adapter trimming was performed using Trim Galore v0.6.7. Raw sequencing data are
deposited at GEO under accession GSE123456. Differential expression was assessed
using DESeq2 v1.38.0. We applied Benjamini-Hochberg correction for multiple testing.
Code is available upon request.
"""

methods_strong = """
RNA-seq reads were trimmed using Trim Galore v0.6.7 (minimum quality score 20,
minimum length 20 bp) and aligned to hg38 using STAR v2.7.10a with parameters
--outSAMtype BAM SortedByCoordinate --outFilterMultimapNmax 1. Read counts were
quantified using featureCounts v2.0.3 (Ensembl v104 annotation). Differential
expression analysis used DESeq2 v1.38.0 in R v4.2.1; Wald test with Benjamini-
Hochberg correction (FDR < 0.05). All stochastic steps used random seed 42.
Raw data are deposited at GEO (GSE234567) and SRA (SRR456789). All analysis
code is available at github.com/lab/rnaseq-pipeline (commit a3f7b12).
"""

checker = MethodsChecker()
results = {}
for name, text in [('Weak', methods_weak), ('Medium', methods_medium), ('Strong', methods_strong)]:
    results[name] = checker.check(text)
    r = results[name]
    print(f"\n=== {name} Methods Section ===")
    print(f"Completeness score: {r['completeness_score']}%")
    print(f"Critical flags: {r['critical_flags']}, Major flags: {r['major_flags']}")
    if r['flags']:
        for f in r['flags']:
            print(f"  [{f.severity.upper()}] {f.category}: {f.message}")
    if r['missing_elements']:
        print("  Missing required elements:")
        for elem, desc in r['missing_elements']:
            print(f"    - {desc}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Methods Section Quality Analysis', fontsize=14, fontweight='bold')

# Panel 1: Radar chart of completeness across 5 categories
categories = ['Software\nVersion', 'Random\nSeed', 'Accession\nNumber', 'Statistical\nTest', 'Code\nRepository']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

ax1 = fig.add_subplot(121, polar=True)
colors_radar = ['#E53935', '#FB8C00', '#43A047']

for (name, r), color in zip(results.items(), colors_radar):
    present_set = set(r['present_elements'])
    all_keys = list(MethodsChecker.REQUIRED_ELEMENTS.keys())
    values = [1 if k in present_set else 0 for k in all_keys]
    values += values[:1]
    ax1.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
    ax1.fill(angles, values, alpha=0.1, color=color)

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, size=9)
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.5, 1])
ax1.set_yticklabels(['Absent', '', 'Present'], size=8)
ax1.set_title('Methods Completeness\n(radar per paper)', pad=20)
ax1.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))

# Panel 2: Flag heatmap
ax2 = axes[1]
flag_categories = ['Parameters', 'Code', 'Methods', 'Data', 'Statistics']
paper_names = ['Weak', 'Medium', 'Strong']
heatmap_data = np.zeros((len(flag_categories), len(paper_names)))

for j, name in enumerate(paper_names):
    for flag in results[name]['flags']:
        if flag.category in flag_categories:
            i = flag_categories.index(flag.category)
            severity_weight = {'critical': 2, 'major': 1, 'minor': 0.5}.get(flag.severity, 1)
            heatmap_data[i, j] += severity_weight

im = ax2.imshow(heatmap_data, cmap='Reds', aspect='auto', vmin=0, vmax=4)
ax2.set_xticks(range(len(paper_names)))
ax2.set_xticklabels(paper_names)
ax2.set_yticks(range(len(flag_categories)))
ax2.set_yticklabels(flag_categories)
ax2.set_title('Flag Severity Heatmap\n(higher = more problems)')
plt.colorbar(im, ax=ax2, label='Weighted severity score')

for i in range(len(flag_categories)):
    for j in range(len(paper_names)):
        val = heatmap_data[i, j]
        if val > 0:
            ax2.text(j, i, f'{val:.1f}', ha='center', va='center',
                     color='white' if val > 2 else 'black', fontweight='bold')

plt.tight_layout()
plt.savefig('methods_checker_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Exercises

**Exercise 1:** Take any paper from Module 08 or 09's `papers.md`. Copy its methods section text. Run it through `MethodsChecker.check()`. What flags are raised?

**Exercise 2:** Find one phrase in a real methods section that qualifies as a red flag. Write a one-sentence explanation of *why* it prevents reproducibility.

**Exercise 3:** Rewrite the `methods_weak` text above to achieve a completeness score of 100% and zero flags. Do not change the biological content — only add the missing information.

**Exercise 4 (reflection):** Why does the presence of a GitHub URL *alone* not guarantee reproducibility? What additional information is needed?

## Quiz

1. Name three red flag phrases and explain what information each is missing.
2. What is the difference between a specification gap and an accessibility gap?
3. Why does software version matter for reproducibility?
4. What is the minimum information needed to find a public dataset mentioned in a methods section?
5. If a methods section says "we used default parameters," what do you need to check before accepting this as reproducible?

## References

- Sandve et al. (2013). "Ten simple rules for reproducible computational research." PLOS Comp Bio 9(10):e1003285.
- Perkel, J.M. (2020). "Make code accessible with these cloud services." Nature 575:247–248.
- Gentleman & Lang (2007). "Statistical analyses and reproducible research." J Comp Graph Stat 16(1):1–23.